# WTA Match Predictor — Modelo ML
Entrenamiento, evaluación y guardado del modelo. Requiere `historico_partidos.csv` y `wta_limpio.csv` generados por el notebook de EDA.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import pickle
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

## 2. Carga de datos

In [ ]:
df = pd.read_csv('historico_partidos.csv', parse_dates=['date'])
print(f'Partidos cargados: {len(df)}')
print(df.info())

## 3. Train / Test split

Split temporal — el test es siempre el período más reciente para simular condiciones reales de predicción.

In [ ]:
corte = '2025-01-01'
df_train = df[df['date'] < corte].copy()
df_test  = df[df['date'] >= corte].copy()

print(f'Train: {len(df_train)} partidos ({df_train["date"].min().year}–{df_train["date"].max().year})')
print(f'Test:  {len(df_test)} partidos ({df_test["date"].min().year}–{df_test["date"].max().year})')

## 4. Tratamiento de outliers

Capeamos `rank_diff` usando los percentiles del train — nunca del test para evitar data leakage.

In [ ]:
limite_superior = df_train['rank_diff'].quantile(0.99)
limite_inferior  = df_train['rank_diff'].quantile(0.01)

df_train['rank_diff'] = df_train['rank_diff'].clip(lower=limite_inferior, upper=limite_superior)
df_test['rank_diff']  = df_test['rank_diff'].clip(lower=limite_inferior, upper=limite_superior)

print(f'Límite inferior (p1%):  {limite_inferior}')
print(f'Límite superior (p99%): {limite_superior}')

## 5. Preparar X e y

In [ ]:
# Columnas que no entran al modelo
cols_excluir = ['match_id', 'date', 'target', 'odd_1', 'odd_2']

X_train = df_train.drop(columns=cols_excluir)
X_test  = df_test.drop(columns=cols_excluir)
y_train = (df_train['target'] == 1).astype(int)
y_test  = (df_test['target'] == 1).astype(int)

print(f'Features: {X_train.columns.tolist()}')
print(f'Shape train: {X_train.shape} | test: {X_test.shape}')

In [ ]:
# Verificar que no hay strings en X
print(X_train.dtypes)
print('\nNulos:')
print(X_train.isnull().sum())

## 6. Pipeline y preprocesador

In [ ]:
cat_cols = ['surface', 'round', 'tournament_type']
num_cols = [c for c in X_train.columns if c not in cat_cols]

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

## 7. Modelos

In [ ]:
# ── Random Forest ─────────────────────────────────────────────────────────────
pipe_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier())
])

rf_params = {
    'classifier': [RandomForestClassifier()],
    'classifier__max_depth': [2, 3, 5],
    'classifier__n_estimators': [100, 200]
}

random_best = GridSearchCV(estimator=pipe_rf, param_grid=rf_params, cv=5, n_jobs=-1)
random_best.fit(X_train, y_train)

print(f'Mejor score RF (CV): {random_best.best_score_:.2%}')
print(f'Mejores parámetros:  {random_best.best_params_}')

In [ ]:
# ── XGBoost ───────────────────────────────────────────────────────────────────
pipe_xgb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(eval_metric='logloss'))
])

xgb_params = {
    'classifier__max_depth': [3, 4, 5, 6],
    'classifier__n_estimators': [100, 200],
    'classifier__learning_rate': [0.05, 0.1, 0.2],
    'classifier__subsample': [0.8, 1.0],
    'classifier__colsample_bytree': [0.8, 1.0]
}

xgb_best = GridSearchCV(estimator=pipe_xgb, param_grid=xgb_params, cv=5, n_jobs=-1)
xgb_best.fit(X_train, y_train)

print(f'Mejor score XGB (CV): {xgb_best.best_score_:.2%}')
print(f'Mejores parámetros:   {xgb_best.best_params_}')

## 8. Evaluación y comparativa

In [ ]:
# Función de evaluación reutilizable
resultados = pd.DataFrame({
    'modelo':            pd.Series(dtype='str'),
    'parametros':        pd.Series(dtype='str'),
    'accuracy_modelo':   pd.Series(dtype='float'),
    'accuracy_casas':    pd.Series(dtype='float'),
    'accuracy_ranking':  pd.Series(dtype='float'),
    'correlacion_casas': pd.Series(dtype='float')
})

def evaluar_modelo(nombre, modelo_fit, X_test, y_test, df_test, df):
    y_pred  = modelo_fit.predict(X_test)
    y_proba = modelo_fit.predict_proba(X_test)[:, 1]

    acc_modelo = accuracy_score(y_test, y_pred)

    # Accuracy casas
    prob_1 = df.loc[df_test['match_id'], 'odd_1'].values
    prob_2 = df.loc[df_test['match_id'], 'odd_2'].values
    mask   = ~np.isnan(prob_1) & ~np.isnan(prob_2)
    pred_casas = (prob_1[mask] > prob_2[mask]).astype(int)
    acc_casas  = accuracy_score(y_test[mask], pred_casas)

    # Accuracy baseline ranking
    pred_ranking = (df_test['rank_diff'] < 0).astype(int)
    acc_ranking  = accuracy_score(y_test, pred_ranking)

    # Correlación con casas
    correlacion = np.corrcoef(y_proba[mask], prob_1[mask])[0, 1]

    try:
        params = modelo_fit.best_params_
    except:
        params = {}

    nueva_fila = {
        'modelo':            nombre,
        'parametros':        str(params),
        'accuracy_modelo':   round(acc_modelo, 4),
        'accuracy_casas':    round(acc_casas, 4),
        'accuracy_ranking':  round(acc_ranking, 4),
        'correlacion_casas': round(correlacion, 4)
    }
    return pd.concat([resultados, pd.DataFrame([nueva_fila])], ignore_index=True)

In [ ]:
resultados = evaluar_modelo('Random Forest', random_best, X_test, y_test, df_test, df)
resultados = evaluar_modelo('XGBoost',       xgb_best,    X_test, y_test, df_test, df)
resultados

## 9. Feature importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, modelo, nombre in [
    (axes[0], random_best, 'Random Forest'),
    (axes[1], xgb_best,    'XGBoost')
]:
    classes      = modelo.best_estimator_.named_steps['preprocessor'].get_feature_names_out()
    importancias = modelo.best_estimator_.named_steps['classifier'].feature_importances_
    df_imp = pd.DataFrame({'feature': classes, 'importancia': importancias})
    df_imp = df_imp.sort_values('importancia', ascending=False)
    sns.barplot(data=df_imp, x='importancia', y='feature', ax=ax)
    ax.set_title(nombre)
    ax.set_xlabel('Importancia')
    ax.set_ylabel('')

plt.tight_layout()
plt.show()

## 10. Guardar el modelo

In [ ]:
# Guardar el mejor modelo (XGBoost) para usar en la app de Streamlit
with open('gbx_red.model', 'wb') as f:
    pickle.dump(xgb_best.best_estimator_, f)

print('Modelo guardado en gbx_red.model')